In [4]:
# Celda 1 — setup y utilidades de path
from pathlib import Path

import geopandas as gpd
import h3
import pandas as pd
from shapely.geometry import Point, Polygon, mapping


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for parent in [start, *start.parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError(f"No se encontró .git subiendo desde {start}")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data" / "ghsl"
OUTPUT_DIR = REPO_ROOT / "outputs" / "qgis_ready"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

H3_RESOLUTION = 9

In [5]:
# Celda 2 — cargar UCDB e inspeccionar columnas (sin asumir nombres de campo)
UCDB_PATH = DATA_DIR / "GHS_UCDB_R2024A.gpkg"  # ajustar al nombre real descargado

ucdb = gpd.read_file(UCDB_PATH).to_crs(epsg=4326)
print(ucdb.columns.tolist())
print(len(ucdb), "urban centres en el archivo")

DataSourceError: /Users/jano/Documents/LInkedin/CoolRefugeEquity_ConcepcionValdivia/data/ghsl/GHS_UCDB_R2024A.gpkg: No such file or directory

In [9]:
UCDB_PATH = DATA_DIR / "GHS_UCDB_REGION_LATIN_AMERICA_AND_THE_CARIBBEAN_R2024A_V1_2" / "GHS_UCDB_REGION_LATIN_AMERICA_AND_THE_CARIBBEAN_R2024A.gpkg"
UCDB_LAYER = "GHSL_UCDB_THEME_GENERAL_CHARACTERISTICS_GLOBE_R2024A"

ucdb = gpd.read_file(UCDB_PATH, layer=UCDB_LAYER)
print("CRS original:", ucdb.crs)
ucdb = ucdb.to_crs(epsg=4326)
print(ucdb.columns.tolist())
print(len(ucdb), "urban centres en Latinoamérica y el Caribe")

CRS original: PROJCS["World_Mollweide",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Mollweide"],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","54009"]]
['ID_UC_G0', 'GC_UCN_MAI_2025', 'GC_UCN_LIS_2025', 'GC_CNT_GAD_2025', 'GC_CNT_UNN_2025', 'GC_UCA_KM2_2025', 'GC_POP_TOT_2025', 'GC_DEV_WIG_2025', 'GC_DEV_USR_2025', 'GC_PLS_SCR_2025', 'GC_UCB_YOB_2025', 'GC_UCB_YOD_2025', 'GC_UCM_CAP', 'geometry']
1089 urban centres en Latinoamérica y el Caribe


In [10]:
# Celda 3 — selección por contención geométrica (no por nombre de texto)
city_points = {
    "concepcion": Point(-73.0444, -36.8201),
    "valdivia": Point(-73.2459, -39.8142),
}
 
city_polygons: dict[str, Polygon] = {}
for city, point in city_points.items():
    match = ucdb[ucdb.contains(point)]
    if len(match) != 1:
        raise ValueError(f"{city}: se esperaba 1 polígono, se encontraron {len(match)}")
    city_polygons[city] = match.geometry.iloc[0]
    print(city, "OK — área UCDB (deg^2, referencial):", match.geometry.iloc[0].area)

concepcion OK — área UCDB (deg^2, referencial): 0.010276641869815112
valdivia OK — área UCDB (deg^2, referencial): 0.0030472423966538403


In [11]:
for city, polygon in city_polygons.items():
    match = ucdb[ucdb.geometry == polygon]
    print(city, "->", match["GC_UCN_MAI_2025"].values[0], "|", match["GC_CNT_UNN_2025"].values[0])

concepcion -> Concepción | Chile
valdivia -> Valdivia | Chile


In [13]:
# Celda 4 — polyfill H3 res 9 por ciudad
def hexagons_for_polygon(polygon: Polygon, resolution: int) -> gpd.GeoDataFrame:
    h3_shape = h3.geo_to_h3shape(mapping(polygon))
    cell_ids = h3.polygon_to_cells(h3_shape, res=resolution)
    records = [
        {"hex_id": cell_id, "geometry": Polygon(
            [(lon, lat) for lat, lon in h3.cell_to_boundary(cell_id)]
        )}
        for cell_id in cell_ids
    ]
    return gpd.GeoDataFrame(records, crs="EPSG:4326")


hex_frames = []
for city, polygon in city_polygons.items():
    hex_gdf = hexagons_for_polygon(polygon, H3_RESOLUTION)
    hex_gdf["city"] = city
    hex_frames.append(hex_gdf)
    print(city, "-", len(hex_gdf), "hexágonos")

hex_grid = gpd.GeoDataFrame(pd.concat(hex_frames, ignore_index=True), crs="EPSG:4326")

concepcion - 1158 hexágonos
valdivia - 314 hexágonos


In [14]:
print(hex_grid.groupby("city").size())

city
concepcion    1158
valdivia       314
dtype: int64


In [15]:
# Celda 5 — reproyectar, validar área y exportar
hex_grid_utm = hex_grid.to_crs(epsg=32718)
hex_grid_utm["area_km2"] = hex_grid_utm.geometry.area / 1e6

print(hex_grid_utm.groupby("city")["area_km2"].agg(["count", "mean", "std"]))
# esperado: mean ~0.105 km2 por hexágono (res 9), std baja y consistente entre ciudades

output_path = OUTPUT_DIR / "hex_grid_res9.gpkg"
hex_grid_utm.to_file(output_path, driver="GPKG")
print("Exportado:", output_path)

            count      mean       std
city                                 
concepcion   1158  0.087882  0.000042
valdivia      314  0.091617  0.000032
Exportado: /Users/jano/Documents/LInkedin/CoolRefugeEquity_ConcepcionValdivia/outputs/qgis_ready/hex_grid_res9.gpkg


In [16]:
# Celda 6 — validación cruzada del área contra h3.cell_area()
hex_grid["area_km2_h3native"] = hex_grid["hex_id"].apply(
    lambda cell_id: h3.cell_area(cell_id, unit="km^2")
)

print(hex_grid.groupby("city")["area_km2_h3native"].agg(["mean", "std"]))

                mean       std
city                          
concepcion  0.087860  0.000044
valdivia    0.091545  0.000032


In [17]:
# Celda 7 — constantes del Componente 2
import ee
import geemap

ee.Initialize()

AOI_BUFFER_M = 500
SEASON_START, SEASON_END = "-01-01", "-03-31"  # enero-marzo

city_years = {
    ("concepcion", 2016): "LANDSAT/LC08/C02/T1_L2",
    ("concepcion", 2026): None,  # L8 + L9 combinados
    ("valdivia", 2016): "LANDSAT/LC08/C02/T1_L2",
    ("valdivia", 2026): None,
}

def city_aoi(city: str) -> ee.Geometry:
    hexes = hex_grid[hex_grid["city"] == city]
    bbox = hexes.total_bounds  # minx, miny, maxx, maxy en EPSG:4326
    geom = ee.Geometry.Rectangle(list(bbox))
    return geom.buffer(AOI_BUFFER_M)

In [18]:
# Celda 8 — máscara de nubes y cálculo de NDVI/LST (Collection 2 Level 2)
def mask_qa_pixel(image: ee.Image) -> ee.Image:
    qa = image.select("QA_PIXEL")
    dilated_cloud = 1 << 1
    cirrus = 1 << 2
    cloud = 1 << 3
    cloud_shadow = 1 << 4
    mask = (
        qa.bitwiseAnd(dilated_cloud).eq(0)
        .And(qa.bitwiseAnd(cirrus).eq(0))
        .And(qa.bitwiseAnd(cloud).eq(0))
        .And(qa.bitwiseAnd(cloud_shadow).eq(0))
    )
    return image.updateMask(mask)


def add_ndvi_lst(image: ee.Image) -> ee.Image:
    ndvi = image.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
    lst_celsius = (
        image.select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("LST")
    )
    return image.addBands([ndvi, lst_celsius])


def build_composite(city: str, year: int) -> ee.Image:
    aoi = city_aoi(city)
    date_start = f"{year}{SEASON_START}"
    date_end = f"{year}{SEASON_END}"

    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi).filterDate(date_start, date_end)
    collection = l8
    if year != 2016:
        l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(aoi).filterDate(date_start, date_end)
        collection = l8.merge(l9)

    processed = collection.map(mask_qa_pixel).map(add_ndvi_lst)
    composite = processed.select(["NDVI", "LST"]).median().clip(aoi)
    return composite

In [27]:
# Celda 8b — reemplaza build_composite: fijar un valor centinela para píxeles sin datos
NODATA_VALUE = -9999.0

def build_composite(city: str, year: int) -> ee.Image:
    aoi = city_aoi(city)
    date_start = f"{year}{SEASON_START}"
    date_end = f"{year}{SEASON_END}"

    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi).filterDate(date_start, date_end)
    collection = l8
    if year != 2016:
        l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(aoi).filterDate(date_start, date_end)
        collection = l8.merge(l9)

    processed = collection.map(mask_qa_pixel).map(add_ndvi_lst)
    composite = processed.select(["NDVI", "LST"]).median().clip(aoi)
    return composite.unmask(NODATA_VALUE)

In [82]:
import inspect
print(inspect.getsource(build_composite))

def build_composite(city: str, year: int) -> ee.Image:
    aoi = city_aoi(city)
    date_start = f"{year}{SEASON_START}"
    date_end = f"{year}{SEASON_END}"

    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi).filterDate(date_start, date_end)
    collection = l8
    if year != 2016:
        l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(aoi).filterDate(date_start, date_end)
        collection = l8.merge(l9)

    processed = collection.map(mask_qa_pixel).map(add_ndvi_lst)
    composite = processed.select(["NDVI", "LST"]).median().clip(aoi)
    return composite.unmask(NODATA_VALUE)



In [83]:
# Celda 9 — reexportar con validación explícita por archivo
def export_composite(city: str, year: int) -> None:
    print(f"Exportando {city} {year}...")
    composite = build_composite(city, year)
    aoi = city_aoi(city)
    filename = f"{city}_{year}_ndvi_lst.tif"
    output_path = OUTPUT_DIR / filename

    if output_path.exists():
        output_path.unlink()  # forzar sobreescritura real, no reuso silencioso

    geemap.ee_export_image(
        composite,
        filename=str(output_path),
        scale=30,
        region=aoi,
        file_per_band=False,
    )

    if not output_path.exists():
        raise RuntimeError(f"{filename}: export falló, el archivo no se creó")

    with rasterio.open(output_path) as src:
        band1 = src.read(1)
        n_sentinel = (band1 == NODATA_VALUE).sum()

    size_mb = output_path.stat().st_size / 1e6
    print(f"  OK — {size_mb:.2f} MB, {n_sentinel} píxeles marcados como nodata")


for (city, year) in city_years:
    export_composite(city, year)

Exportando curico 2016...
Generating URL ...
Please wait ...
Data downloaded to /Users/jano/Documents/LInkedin/CoolRefugeEquity_ConcepcionValdivia/outputs/qgis_ready/curico_2016_ndvi_lst.tif
  OK — 0.78 MB, 0 píxeles marcados como nodata
Exportando curico 2026...
Generating URL ...
Please wait ...
Data downloaded to /Users/jano/Documents/LInkedin/CoolRefugeEquity_ConcepcionValdivia/outputs/qgis_ready/curico_2026_ndvi_lst.tif
  OK — 0.80 MB, 0 píxeles marcados como nodata
Exportando valdivia 2016...
Generating URL ...
Please wait ...
Data downloaded to /Users/jano/Documents/LInkedin/CoolRefugeEquity_ConcepcionValdivia/outputs/qgis_ready/valdivia_2016_ndvi_lst.tif
  OK — 0.85 MB, 0 píxeles marcados como nodata
Exportando valdivia 2026...
Generating URL ...
Please wait ...
Data downloaded to /Users/jano/Documents/LInkedin/CoolRefugeEquity_ConcepcionValdivia/outputs/qgis_ready/valdivia_2026_ndvi_lst.tif
  OK — 0.86 MB, 0 píxeles marcados como nodata


In [29]:
# Celda 10 — setup Componente 3
import numpy as np
import rasterio
from rasterstats import zonal_stats

MIN_VALID_FRACTION = 0.5

raster_paths = {
    (city, year): OUTPUT_DIR / f"{city}_{year}_ndvi_lst.tif"
    for (city, year) in city_years
}

In [84]:
# Celda 11 — agregación zonal, umbral de cobertura aplicado por banda (reemplaza la Celda 11 completa)
def zonal_aggregate(city: str, year: int, hex_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    raster_path = raster_paths[(city, year)]

    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        pixel_area_m2 = abs(src.transform.a * src.transform.e)

    hex_local = hex_gdf[hex_gdf["city"] == city].to_crs(raster_crs).reset_index(drop=True)

    stats_ndvi = zonal_stats(hex_local, str(raster_path), band=1, stats=["mean", "count"], nodata=NODATA_VALUE)
    stats_lst = zonal_stats(hex_local, str(raster_path), band=2, stats=["mean", "count"], nodata=NODATA_VALUE)

    records = []
    for i, row in hex_local.iterrows():
        n_theoretical = row.geometry.area / pixel_area_m2

        valid_fraction_ndvi = (stats_ndvi[i]["count"] or 0) / n_theoretical if n_theoretical > 0 else 0.0
        valid_fraction_lst = (stats_lst[i]["count"] or 0) / n_theoretical if n_theoretical > 0 else 0.0

        ndvi_mean = stats_ndvi[i]["mean"] if valid_fraction_ndvi >= MIN_VALID_FRACTION else np.nan
        lst_mean = stats_lst[i]["mean"] if valid_fraction_lst >= MIN_VALID_FRACTION else np.nan

        records.append({
            "hex_id": row["hex_id"],
            "city": city,
            "year": year,
            "ndvi_mean": ndvi_mean,
            "lst_mean": lst_mean,
            "valid_fraction_ndvi": round(valid_fraction_ndvi, 3),
            "valid_fraction_lst": round(valid_fraction_lst, 3),
        })
    return pd.DataFrame(records)

In [85]:
# Celda 12 — correr para las 4 combinaciones y revisar cobertura
zonal_frames = [zonal_aggregate(city, year, hex_grid) for (city, year) in city_years]
metrics = pd.concat(zonal_frames, ignore_index=True)

n_dropped_ndvi = metrics["ndvi_mean"].isna().sum()
n_dropped_lst = metrics["lst_mean"].isna().sum()
print(f"{n_dropped_ndvi} hexágonos sin NDVI válido, {n_dropped_lst} sin LST válido (umbral {MIN_VALID_FRACTION:.0%})")
print(metrics.groupby(["city", "year"])[["valid_fraction_ndvi", "valid_fraction_lst"]].describe())

Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as 

0 hexágonos sin NDVI válido, 32 sin LST válido (umbral 50%)
              valid_fraction_ndvi                                           \
                            count      mean       std    min    25%    50%   
city     year                                                                
curico   2016               251.0  0.999283  0.028211  0.965  0.981  0.989   
         2026               251.0  0.999283  0.028211  0.965  0.981  0.989   
valdivia 2016               314.0  1.000083  0.018667  0.971  0.980  1.001   
         2026               314.0  1.000083  0.018667  0.971  0.980  1.001   

                            valid_fraction_lst                             \
                 75%    max              count      mean       std    min   
city     year                                                               
curico   2016  1.031  1.057              251.0  0.999283  0.028211  0.965   
         2026  1.031  1.057              251.0  0.999283  0.028211  0.965   
valdivia

In [38]:
metrics["z_ndvi"] = metrics.groupby(["city", "year"])["ndvi_mean"].transform(zscore)
metrics["z_lst"] = metrics.groupby(["city", "year"])["lst_mean"].transform(zscore)
metrics["indice_refresco"] = metrics["z_ndvi"] - metrics["z_lst"]

print(metrics.groupby(["city", "year"])[["z_ndvi", "z_lst"]].agg(["mean", "std"]))

                       z_ndvi              z_lst     
                         mean  std          mean  std
city       year                                      
concepcion 2016  1.159119e-16  1.0 -1.746476e-16  1.0
           2026 -7.248088e-17  1.0  2.578855e-16  1.0
valdivia   2016  1.085473e-16  1.0  1.131464e-15  1.0
           2026 -7.990777e-17  1.0 -2.592300e-16  1.0


In [55]:
# Celda 12b (repetir) — confirmar que el nodata ahora está presente
for (city, year), path in raster_paths.items():
    with rasterio.open(path) as src:
        print(f"{city} {year}: nodata={src.nodata}, dtype={src.dtypes}")
        band1 = src.read(1)
        n_sentinel = (band1 == -9999.0).sum()
        print(f"  min={band1.min():.4f}, max={band1.max():.4f}, "
              f"n_pixeles_-9999={n_sentinel} de {band1.size}")

concepcion 2016: nodata=None, dtype=('float64', 'float64')
  min=-9999.0000, max=0.5379, n_pixeles_-9999=223 de 401568
concepcion 2026: nodata=None, dtype=('float64', 'float64')
  min=-0.0913, max=0.5401, n_pixeles_-9999=0 de 401568
valdivia 2016: nodata=None, dtype=('float64', 'float64')
  min=-0.1185, max=0.5504, n_pixeles_-9999=0 de 101697
valdivia 2026: nodata=None, dtype=('float64', 'float64')
  min=-0.0453, max=0.5454, n_pixeles_-9999=0 de 101697


In [56]:
# Celda 12c — ¿los rasters de 2016 y 2026 son realmente distintos?
import hashlib

for city in ["concepcion", "valdivia"]:
    hashes = {}
    for year in [2016, 2026]:
        with open(raster_paths[(city, year)], "rb") as f:
            hashes[year] = hashlib.md5(f.read()).hexdigest()
    print(city, "2016 == 2026:", hashes[2016] == hashes[2026])

concepcion 2016 == 2026: False
valdivia 2016 == 2026: False


In [86]:
# Celda 13 — z-score intra-ciudad-año
metrics["z_ndvi"] = metrics.groupby(["city", "year"])["ndvi_mean"].transform(zscore)
metrics["z_lst"] = metrics.groupby(["city", "year"])["lst_mean"].transform(zscore)
metrics["indice_refresco"] = metrics["z_ndvi"] - metrics["z_lst"]

print(metrics.groupby(["city", "year"])[["z_ndvi", "z_lst"]].agg(["mean", "std"]))

                     z_ndvi              z_lst     
                       mean  std          mean  std
city     year                                      
curico   2016 -4.157807e-16  1.0  8.625239e-18  1.0
         2026 -1.105800e-17  1.0  1.855090e-15  1.0
valdivia 2016  1.085473e-16  1.0 -6.437803e-16  1.0
         2026 -7.990777e-17  1.0 -1.122704e-15  1.0


In [87]:
# Celda 14 — unir a geometría y exportar (sobrescribe hex_indice_refresco.gpkg con datos de Curicó)
hex_metrics = hex_grid_utm.drop(columns="area_km2").merge(
    metrics, on=["hex_id", "city"], how="left"
)

output_path = OUTPUT_DIR / "hex_indice_refresco.gpkg"
hex_metrics.to_file(output_path, driver="GPKG")
print("Exportado:", output_path)

Exportado: /Users/jano/Documents/LInkedin/CoolRefugeEquity_ConcepcionValdivia/outputs/qgis_ready/hex_indice_refresco.gpkg


In [59]:
# Celda 15 — inspeccionar la colección antes de asumir convención de filtrado por año
worldpop = ee.ImageCollection("projects/sat-io/open-datasets/WORLDPOP/pop")

print("N° de imágenes en la colección:", worldpop.size().getInfo())

first_image = worldpop.first()
print("Bandas:", first_image.bandNames().getInfo())
print("Propiedades:", first_image.propertyNames().getInfo())
print("system:time_start ->", first_image.get("system:time_start").getInfo())
print("system:index ->", first_image.get("system:index").getInfo())

N° de imágenes en la colección: 3870
Bandas: ['population']
Propiedades: ['country', 'year', 'release', 'system:time_end', 'release_year', 'system:id', 'source', 'type', 'resolution', 'temporal_resolution', 'version', 'system:time_start', 'country_code', 'system:footprint', 'system:version', 'publication_date', 'collection_description', 'created_date', 'system:asset_size', 'dataset', 'spatial_resolution', 'system:index', 'system:bands', 'system:band_names']
system:time_start -> 1420070400000
system:index -> ABW_POP_2015_CN_100M_R2025A_V1


In [60]:
# Celda 16 — confirmar tipo de dato de 'year' y 'country_code' antes de filtrar
sample = worldpop.first()
print("year (tipo/valor):", sample.get("year").getInfo())
print("country_code (tipo/valor):", sample.get("country_code").getInfo())

chile_test = worldpop.filter(ee.Filter.eq("country_code", "CHL"))
print("N° imágenes Chile:", chile_test.size().getInfo())
print("Años disponibles para Chile:", chile_test.aggregate_array("year").getInfo())


year (tipo/valor): 2015
country_code (tipo/valor): ABW
N° imágenes Chile: 16
Años disponibles para Chile: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026', '2027', '2028', '2029', '2030']


In [88]:
# Celda 17 — función de agregación de población por hexágono (suma ponderada por área)
def population_for_city_year(city: str, year: int, hex_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    hex_local = hex_gdf[hex_gdf["city"] == city][["hex_id", "geometry"]].to_crs(epsg=4326)
    hex_fc = geemap.geopandas_to_ee(hex_local)

    pop_image = (
        worldpop
        .filter(ee.Filter.eq("country_code", "CHL"))
        .filter(ee.Filter.eq("year", str(year)))
        .first()
    )

    result_fc = pop_image.reduceRegions(
        collection=hex_fc,
        reducer=ee.Reducer.sum(),
        scale=100,
    )

    features = result_fc.getInfo()["features"]
    records = [
        {
            "hex_id": f["properties"]["hex_id"],
            "city": city,
            "year": year,
            "poblacion": f["properties"].get("sum", 0.0),
        }
        for f in features
    ]
    return pd.DataFrame(records)

In [89]:
# Celda 18 — correr para las 4 combinaciones
pop_frames = [population_for_city_year(city, year, hex_grid) for (city, year) in city_years]
population_df = pd.concat(pop_frames, ignore_index=True)

print(population_df.groupby(["city", "year"])["poblacion"].sum())

city      year
curico    2016     87331.363770
          2026     95417.677935
valdivia  2016    120023.581457
          2026    131270.052746
Name: poblacion, dtype: float64


In [90]:
# Celda 19 — chequeo de orden de magnitud contra UCDB
for city, polygon in city_polygons.items():
    match = ucdb[ucdb.geometry == polygon]
    print(city, "GC_POP_TOT_2025 (UCDB):", match["GC_POP_TOT_2025"].values[0])

print()
print("Suma de hexágonos, por ciudad-año (WorldPop):")
print(population_df.groupby(["city", "year"])["poblacion"].sum())

curico GC_POP_TOT_2025 (UCDB): 103446.7394
valdivia GC_POP_TOT_2025 (UCDB): 148826.0266

Suma de hexágonos, por ciudad-año (WorldPop):
city      year
curico    2016     87331.363770
          2026     95417.677935
valdivia  2016    120023.581457
          2026    131270.052746
Name: poblacion, dtype: float64


In [91]:
# Celda 20 — fusionar población con hex_indice_refresco.gpkg existente
hex_metrics_path = OUTPUT_DIR / "hex_indice_refresco.gpkg"
hex_metrics = gpd.read_file(hex_metrics_path)

hex_metrics = hex_metrics.merge(population_df, on=["hex_id", "city", "year"], how="left")

n_missing_pop = hex_metrics["poblacion"].isna().sum()
print(f"{n_missing_pop} hexágonos sin dato de población")

hex_metrics.to_file(hex_metrics_path, driver="GPKG")
print("Actualizado:", hex_metrics_path)

0 hexágonos sin dato de población
Actualizado: /Users/jano/Documents/LInkedin/CoolRefugeEquity_ConcepcionValdivia/outputs/qgis_ready/hex_indice_refresco.gpkg


In [92]:
# Celda 21 — cuartil de Índice de Refresco, intra-ciudad-año
def assign_quartile(series: pd.Series) -> pd.Series:
    return pd.qcut(series, q=4, labels=[1, 2, 3, 4])


hex_metrics["cuartil_refresco"] = hex_metrics.groupby(["city", "year"])["indice_refresco"].transform(assign_quartile)

# cuartil 1 = 25% de hexágonos con peor Índice de Refresco (más calor/menos verde relativo)
print(hex_metrics.groupby(["city", "year", "cuartil_refresco"]).size())

city      year  cuartil_refresco
curico    2016  1                   63
                2                   63
                3                   62
                4                   63
          2026  1                   63
                2                   63
                3                   62
                4                   63
valdivia  2016  1                   75
                2                   74
                3                   74
                4                   75
          2026  1                   75
                2                   74
                3                   74
                4                   75
dtype: int64


/var/folders/c2/103818s13slfkty61zncfy9c0000gn/T/ipykernel_88955/2743564500.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(hex_metrics.groupby(["city", "year", "cuartil_refresco"]).size())


In [93]:
# Celda 22 — % de población en el peor cuartil, por ciudad-año
def poblacion_en_peor_cuartil(df: pd.DataFrame) -> pd.Series:
    total = df["poblacion"].sum()
    peor_cuartil = df.loc[df["cuartil_refresco"] == 1, "poblacion"].sum()
    return pd.Series({
        "poblacion_total": total,
        "poblacion_peor_cuartil": peor_cuartil,
        "pct_poblacion_peor_cuartil": 100 * peor_cuartil / total,
    })


equidad = hex_metrics.groupby(["city", "year"]).apply(poblacion_en_peor_cuartil, include_groups=False)
print(equidad)

               poblacion_total  poblacion_peor_cuartil  \
city     year                                            
curico   2016     87331.363770            22903.245102   
         2026     95417.677935            26875.918801   
valdivia 2016    120023.581457            50904.811314   
         2026    131270.052746            55081.842741   

               pct_poblacion_peor_cuartil  
city     year                              
curico   2016                   26.225681  
         2026                   28.166603  
valdivia 2016                   42.412342  
         2026                   41.960707  


In [94]:
# Celda 23 — comparación 2016 vs 2026 por ciudad
equidad_wide = equidad["pct_poblacion_peor_cuartil"].unstack("year")
equidad_wide["delta_2016_2026"] = equidad_wide[2026] - equidad_wide[2016]
print(equidad_wide)

year           2016       2026  delta_2016_2026
city                                           
curico    26.225681  28.166603         1.940922
valdivia  42.412342  41.960707        -0.451634


In [95]:
# Celda 24 — exportar tabla de equidad final
equidad_output_path = OUTPUT_DIR / "equidad_termica.csv"
equidad.reset_index().to_csv(equidad_output_path, index=False)
print("Exportado:", equidad_output_path)

Exportado: /Users/jano/Documents/LInkedin/CoolRefugeEquity_ConcepcionValdivia/outputs/qgis_ready/equidad_termica.csv


In [96]:
# Celda 25 — reexportar el gpkg con cuartil_refresco incluido
hex_metrics.to_file(hex_metrics_path, driver="GPKG")
print("Actualizado:", hex_metrics_path)

Actualizado: /Users/jano/Documents/LInkedin/CoolRefugeEquity_ConcepcionValdivia/outputs/qgis_ready/hex_indice_refresco.gpkg


In [72]:
# Celda 26 — diagnóstico de huecos en cuartil_refresco por ciudad-año (corregido)
print(hex_metrics.groupby(["city", "year"])["cuartil_refresco"].apply(lambda s: s.isna().sum()))
print()
nan_cuartil = hex_metrics[hex_metrics["cuartil_refresco"].isna()]
print("Hexágonos con cuartil NaN:", len(nan_cuartil))
if len(nan_cuartil) > 0:
    print(nan_cuartil[["city", "year", "indice_refresco", "ndvi_mean", "lst_mean", "valid_fraction_ndvi", "valid_fraction_lst"]])

city        year
concepcion  2016    579
            2026    567
valdivia    2016     16
            2026     16
Name: cuartil_refresco, dtype: int64

Hexágonos con cuartil NaN: 1178
            city  year  indice_refresco  ndvi_mean  lst_mean  \
4     concepcion  2016              NaN   0.075352       NaN   
5     concepcion  2026              NaN   0.119638       NaN   
6     concepcion  2016              NaN   0.073044       NaN   
7     concepcion  2026              NaN   0.080473       NaN   
12    concepcion  2016              NaN   0.257265       NaN   
...          ...   ...              ...        ...       ...   
2819    valdivia  2026              NaN   0.304530       NaN   
2826    valdivia  2016              NaN   0.245716       NaN   
2827    valdivia  2026              NaN   0.277613       NaN   
2830    valdivia  2016              NaN   0.289688       NaN   
2831    valdivia  2026              NaN   0.296722       NaN   

      valid_fraction_ndvi  valid_fraction_lst  


In [73]:
# Celda 27 — diagnóstico: ¿el conteo de píxeles válidos difiere entre NDVI y LST?
def diagnostic_zonal(city: str, year: int, hex_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    raster_path = raster_paths[(city, year)]
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        pixel_area_m2 = abs(src.transform.a * src.transform.e)

    hex_local = hex_gdf[hex_gdf["city"] == city].to_crs(raster_crs).reset_index(drop=True)
    stats_ndvi = zonal_stats(hex_local, str(raster_path), band=1, stats=["count"], nodata=NODATA_VALUE)
    stats_lst = zonal_stats(hex_local, str(raster_path), band=2, stats=["count"], nodata=NODATA_VALUE)

    return pd.DataFrame({
        "hex_id": hex_local["hex_id"],
        "count_ndvi": [s["count"] for s in stats_ndvi],
        "count_lst": [s["count"] for s in stats_lst],
    })


diag = diagnostic_zonal("concepcion", 2016, hex_grid)
diag["diff"] = diag["count_ndvi"] - diag["count_lst"]
print(diag["diff"].describe())
print("Hexágonos donde LST tiene menos píxeles válidos que NDVI:", (diag["diff"] > 0).sum())

Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


count    1158.000000
mean       61.321244
std        52.302528
min         0.000000
25%         0.000000
50%        60.500000
75%       121.000000
max       126.000000
Name: diff, dtype: float64
Hexágonos donde LST tiene menos píxeles válidos que NDVI: 837


In [74]:
# Celda 28 — diagnóstico: ¿qué bit de QA_PIXEL está causando la pérdida de cobertura en LST?
def diagnose_qa_bits(city: str, year: int) -> None:
    aoi = city_aoi(city)
    date_start = f"{year}{SEASON_START}"
    date_end = f"{year}{SEASON_END}"
    collection = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi).filterDate(date_start, date_end)

    def count_valid_st(image, bits_to_check):
        qa = image.select("QA_PIXEL")
        mask = ee.Image.constant(1)
        for bit in bits_to_check:
            mask = mask.And(qa.bitwiseAnd(1 << bit).eq(0))
        return image.select("ST_B10").updateMask(mask).mask()

    scenarios = {
        "todos_los_bits (actual)": [1, 2, 3, 4],
        "sin_cloud_shadow": [1, 2, 3],
        "solo_cloud": [3],
        "sin_mascara": [],
    }

    for label, bits in scenarios.items():
        masked = collection.map(lambda img: count_valid_st(img, bits))
        composite_mask = masked.max()
        valid_fraction = composite_mask.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=aoi, scale=30, maxPixels=1e9
        ).get("ST_B10").getInfo()
        print(f"{label}: {valid_fraction:.3f} fracción del AOI con al menos 1 pasada válida")


diagnose_qa_bits("concepcion", 2016)

todos_los_bits (actual): 0.613 fracción del AOI con al menos 1 pasada válida
sin_cloud_shadow: 0.613 fracción del AOI con al menos 1 pasada válida
solo_cloud: 0.613 fracción del AOI con al menos 1 pasada válida
sin_mascara: 0.613 fracción del AOI con al menos 1 pasada válida


In [75]:
# Celda 29 — diagnóstico: ¿cuántas imágenes hay y cubren realmente todo el AOI?
def diagnose_scene_coverage(city: str, year: int) -> None:
    aoi = city_aoi(city)
    date_start = f"{year}{SEASON_START}"
    date_end = f"{year}{SEASON_END}"
    collection = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi).filterDate(date_start, date_end)

    n_images = collection.size().getInfo()
    print(f"N° de escenas Landsat 8 que intersectan el AOI: {n_images}")

    dates = collection.aggregate_array("system:time_start").getInfo()
    import datetime
    print("Fechas de adquisición:")
    for d in sorted(dates):
        print(" ", datetime.datetime.utcfromtimestamp(d / 1000).strftime("%Y-%m-%d"))

    path_rows = collection.aggregate_array("WRS_PATH").getInfo(), collection.aggregate_array("WRS_ROW").getInfo()
    print("Combinaciones PATH/ROW:", set(zip(*path_rows)))

    union_footprint = collection.geometry()
    coverage_fraction = ee.Image.constant(1).clip(union_footprint).reduceRegion(
        reducer=ee.Reducer.count(), geometry=aoi, scale=30, maxPixels=1e9
    ).get("constant").getInfo()
    aoi_pixel_count = ee.Image.constant(1).reduceRegion(
        reducer=ee.Reducer.count(), geometry=aoi, scale=30, maxPixels=1e9
    ).get("constant").getInfo()
    print(f"Fracción del AOI cubierta por AL MENOS una escena: {coverage_fraction / aoi_pixel_count:.3f}")


diagnose_scene_coverage("concepcion", 2016)

N° de escenas Landsat 8 que intersectan el AOI: 12
Fechas de adquisición:
  2016-01-06
  2016-01-06
  2016-01-22
  2016-01-22
  2016-02-07
  2016-02-07
  2016-02-23
  2016-02-23
  2016-03-10
  2016-03-10
  2016-03-26
  2016-03-26
Combinaciones PATH/ROW: {(1, 85), (1, 86)}
Fracción del AOI cubierta por AL MENOS una escena: 1.000


In [76]:
# Celda 30 — validar cobertura ST con ventana ampliada, antes de reexportar todo
def diagnose_scene_coverage_v2(city: str, year: int, date_start: str, date_end: str) -> float:
    aoi = city_aoi(city)
    collection = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi).filterDate(date_start, date_end)

    n_images = collection.size().getInfo()

    st_valid_mask = collection.select("ST_B10").map(lambda img: img.mask()).max()
    coverage = st_valid_mask.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=30, maxPixels=1e9
    ).get("ST_B10").getInfo()

    print(f"{city} {year}: {n_images} escenas, cobertura ST válida = {coverage:.3f}")
    return coverage


# ventana actual (ene-mar) vs ampliada (dic-abr)
print("--- Ventana actual (ene-mar) ---")
diagnose_scene_coverage_v2("concepcion", 2016, "2016-01-01", "2016-03-31")

print("--- Ventana ampliada (dic-abr) ---")
diagnose_scene_coverage_v2("concepcion", 2016, "2015-12-01", "2016-04-30")

--- Ventana actual (ene-mar) ---
concepcion 2016: 12 escenas, cobertura ST válida = 0.613
--- Ventana ampliada (dic-abr) ---
concepcion 2016: 20 escenas, cobertura ST válida = 0.614


0.6143047564916583

In [77]:
# Celda 31 — diagnóstico rápido de cobertura ST en Curicó, sin construir el pipeline completo
curico_point = ee.Geometry.Point([-71.2394, -34.9828])
curico_aoi = curico_point.buffer(5000)  # AOI provisional, solo para el diagnóstico

collection_curico = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(curico_aoi)
    .filterDate("2016-01-01", "2016-03-31")
)

n_images = collection_curico.size().getInfo()
print(f"N° de escenas Landsat 8 (Curicó, ene-mar 2016): {n_images}")

st_valid_mask = collection_curico.select("ST_B10").map(lambda img: img.mask()).max()
coverage = st_valid_mask.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=curico_aoi, scale=30, maxPixels=1e9
).get("ST_B10").getInfo()
print(f"Cobertura ST válida en Curicó: {coverage:.3f}")

N° de escenas Landsat 8 (Curicó, ene-mar 2016): 5
Cobertura ST válida en Curicó: 1.000


In [78]:
# Celda 32 — reemplazar Concepción por Curicó en la selección de ciudades
city_points = {
    "curico": Point(-71.2394, -34.9828),
    "valdivia": Point(-73.2459, -39.8142),
}

city_polygons: dict[str, Polygon] = {}
for city, point in city_points.items():
    match = ucdb[ucdb.contains(point)]
    if len(match) != 1:
        raise ValueError(f"{city}: se esperaba 1 polígono, se encontraron {len(match)}")
    city_polygons[city] = match.geometry.iloc[0]
    print(city, "->", match["GC_UCN_MAI_2025"].values[0], "|", match["GC_CNT_UNN_2025"].values[0])

curico -> Curicó | Chile
valdivia -> Valdivia | Chile


In [79]:
# Celda 33 — regenerar malla H3 solo para Curicó (Valdivia no se toca)
hex_frames_new = []
for city, polygon in city_polygons.items():
    hex_gdf = hexagons_for_polygon(polygon, H3_RESOLUTION)
    hex_gdf["city"] = city
    hex_frames_new.append(hex_gdf)
    print(city, "-", len(hex_gdf), "hexágonos")

hex_grid = gpd.GeoDataFrame(pd.concat(hex_frames_new, ignore_index=True), crs="EPSG:4326")

curico - 251 hexágonos
valdivia - 314 hexágonos


In [80]:
# Celda 34 — validar área, exportar la malla nueva (sobrescribe hex_grid_res9.gpkg)
hex_grid_utm = hex_grid.to_crs(epsg=32718)
hex_grid_utm["area_km2"] = hex_grid_utm.geometry.area / 1e6
print(hex_grid_utm.groupby("city")["area_km2"].agg(["count", "mean", "std"]))

output_path = OUTPUT_DIR / "hex_grid_res9.gpkg"
hex_grid_utm.to_file(output_path, driver="GPKG")
print("Exportado:", output_path)

          count      mean       std
city                               
curico      251  0.088640  0.000022
valdivia    314  0.091617  0.000032
Exportado: /Users/jano/Documents/LInkedin/CoolRefugeEquity_ConcepcionValdivia/outputs/qgis_ready/hex_grid_res9.gpkg


In [81]:
# Celda 35 — actualizar combinaciones ciudad-año (Curicó reemplaza Concepción)
city_years = {
    ("curico", 2016): "LANDSAT/LC08/C02/T1_L2",
    ("curico", 2026): None,
    ("valdivia", 2016): "LANDSAT/LC08/C02/T1_L2",
    ("valdivia", 2026): None,
}

raster_paths = {
    (city, year): OUTPUT_DIR / f"{city}_{year}_ndvi_lst.tif"
    for (city, year) in city_years
}